# Test `generate_building_boundary` and prepare a Grasshopper handoff

This notebook does three things:
1. Runs the local Python footprint generator.
2. Inspects the returned boundary coordinates and metrics.
3. Builds a JSON payload that a Grasshopper-side import tool can consume through Swiftlet MCP.

## Expected handoff architecture

Keep the initial footprint generator in Python. Then add a separate Grasshopper import tool that accepts polygon coordinates and creates a Rhino polyline or curve.

Recommended data contract:
```json
{
  "geometry_id": "generate_building_boundary_xxx",
  "building_footprint": {
    "type": "Polygon",
    "coordinates": [[x, y, z], [x, y, z], ...]
  },
  "metadata": {
    "shape_type": "I",
    "boundary_area_sqm": 900.0
  }
}
```

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

from agent.tools.generate_building_boundary import generate_building_boundary

NOTEBOOK_ROOT = Path.cwd()
OUTPUT_JSON = NOTEBOOK_ROOT / 'generated_building_boundary_payload.json'
SWIFTLET_MCP_URL = 'http://localhost:3001/mcp/'
GH_IMPORT_TOOL_NAME = 'import_building_boundary_04'

In [ ]:
result = generate_building_boundary(
    area=1200.0,
    building_type='L',
    building_depth=18.0,
    shape_ratio=0.62,
    location_xy=(30.0, 20.0),
    is_mirrored=False,
    max_rotation_angle=180.0,
    max_rotation_step=4,
    rotation_step=1,
)

result

In [ ]:
boundary = result['data']['boundary']
boundary[:5], len(boundary), result['data']['boundary_area_sqm'], result['data']['perimeter_m']

In [ ]:
gh_payload = {
    'geometry_id': result['data']['geometry_id'],
    'building_footprint': {
        'type': 'Polygon',
        'coordinates': result['data']['boundary'],
    },
    'metadata': {
        'shape_type': result['data']['shape_type'],
        'boundary_area_sqm': result['data']['boundary_area_sqm'],
        'perimeter_m': result['data']['perimeter_m'],
        'centroid': result['data']['centroid'],
        'bounding_box': result['data']['bounding_box'],
        'generator_parameters': result['data']['parameters'],
        'source_tool': result['metadata']['tool_name'],
    },
}

OUTPUT_JSON.write_text(json.dumps(gh_payload, indent=2), encoding='utf-8')
gh_payload

## Two ways to hand the boundary to Grasshopper

### Option A: MCP import tool
Best option once you add a Grasshopper-side tool such as `import_building_boundary_04`. That tool should:
- accept the polygon coordinates from `gh_payload`
- create a closed Rhino polyline or Nurbs curve
- optionally bake it to a named layer
- return Rhino GUIDs and any derived metrics

### Option B: File watcher in Grasshopper
Use the JSON file written above. Grasshopper can read it with a File Path component + JSON parser or a GhPython/C# component, then reconstruct the polyline from the coordinates.

In [ ]:
def build_mcp_tool_call_payload(tool_name: str, arguments: dict) -> dict:
    return {
        'jsonrpc': '2.0',
        'id': 1,
        'method': 'tools/call',
        'params': {
            'name': tool_name,
            'arguments': arguments,
        },
    }

mcp_arguments = {
    'geometry_id': gh_payload['geometry_id'],
    'boundary': gh_payload['building_footprint']['coordinates'],
    'shape_type': gh_payload['metadata']['shape_type'],
    'layer_name': 'TerraPilot_Output::BuildingFootprint',
    'closed': True,
}

mcp_request_body = build_mcp_tool_call_payload(GH_IMPORT_TOOL_NAME, mcp_arguments)
mcp_request_body

In [ ]:
# Run this cell only after Rhino + Swiftlet are open and the Grasshopper import tool exists.
# If the tool is not implemented yet, this request will fail, which is expected.

import urllib.error
import urllib.request

request = urllib.request.Request(
    SWIFTLET_MCP_URL,
    data=json.dumps(mcp_request_body).encode('utf-8'),
    headers={'Content-Type': 'application/json'},
    method='POST',
)

try:
    with urllib.request.urlopen(request, timeout=30) as response:
        bridge_result = json.loads(response.read().decode('utf-8'))
    bridge_result
except urllib.error.URLError as exc:
    print('Swiftlet MCP request failed:', exc)
    print('If Rhino is open, the likely missing piece is the Grasshopper import tool implementation.')

## Grasshopper-side import tool sketch

Your Grasshopper bridge tool should accept `boundary` as a list of `[x, y, z]` points, convert them into Rhino points, create a closed polyline, and optionally bake it.

Minimum GH tool inputs:
- `geometry_id`: string
- `boundary`: array of `[x, y, z]` points
- `shape_type`: string
- `layer_name`: string
- `closed`: boolean

Minimum GH tool outputs:
- `geometry_id`: string
- `footprint_guid`: Rhino curve GUID
- `point_count`: integer
- `is_closed`: boolean
- `layer_name`: string